# Exploring Encodec

This notebook:
- loads HF `facebook/encodec_24khz`
- reads `.ecdc` (via your `utils.ecdc_utils.load_ecdc`)
- reads audio files and can code them as Encodec token staks or latents  
- builds a per-level lookup table **by decoding each token index** (RNeNcodec-style)  
--      (this is used to map token stacks (or individual tokens) to latents



In [1]:
import torch
import torch.nn.functional as F
import numpy as np
from transformers import EncodecModel

from IPython.display import Audio, display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "facebook/encodec_24khz"

model = EncodecModel.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()

print("Loaded:", MODEL_ID)
print("Device:", DEVICE)
print("Model sampling rate:", getattr(model.config, "sampling_rate", None))
print("Codebook size (K):", getattr(model.config, "codebook_size", None))
print("Target bandwidths:", getattr(model.config, "target_bandwidths", None))

Loaded: facebook/encodec_24khz
Device: cpu
Model sampling rate: 24000
Codebook size (K): 1024
Target bandwidths: [1.5, 3.0, 6.0, 12.0, 24.0]


In [2]:
from utils.ecdc_utils import load_wav_mono
from utils.ecdc_utils import load_ecdc  # read a stored ecdc file
from utils.ecdc_utils import  encode_audio_to_tokens #convert an audio to an ecdc representation

from utils.ecdc_utils import bandwidth_to_n_q, n_q_to_bandwidth

# Creates the lookup table for an Encodec model to use for token-> latent mapping
from utils.ecdc_utils import  build_LOOKUP_via_layer_decode

# Creates a "pool" of tokens that can be drawn on for style transfer purposes
from utils.ecdc_utils import  tokens_to_summary_latents

# returns tensor of latents for just one level of a token stack sequence
from utils.ecdc_utils import  token_level_to_latents

# take a target audio segment to a tensor of encodec latents
from utils.ecdc_utils import  audio_to_latents

from utils.ecdc_utils import  latents128_to_audio, tokens_TN_to_audio_1T
#=======================
# imports specifically for the Tokui-like style transfer


<div style="width: 100%; height: 2px; background-color: red;"></div>
Tensor ordering is a mess between HF (Hugging Face, Encodec, and canonical). <br>
<span style="color: red;">
      - "TN":  [T_frames, n_q]      (recommended canonical)  <br>  
      - "BQT": [B, n_q, T_frames]   (HuggingFace-friendly)  <br>  
      - "QBT": [n_q, B, T_frames]   (for model.quantizer.decode)  <br>  
</span>

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>Utility Functions for your coding pleasure</b><br>

<span style="color: blue;">
Translate between Encodec "bandwidth" representations
bandwidths (kbs): [1.5, 3.0, 6.0, 12.0, 24.0] <->  number of codebooks (n_q): [2, 4, 8, 16, 32]  
</span> <br>
bandwidth_to_n_q(bw_kbps)  
n_q_to_bandwidth(n_q)   
<br><br>

<span style="color: blue;">
Builds the datastructure for mapping tokens to latents   
</span> <br>
build_LOOKUP_via_layer_decode(model, n_q, K_codbook_size, device=DEVICE)
<br><br>

<span style="color: blue;">
Uses the loopkup table datastructure to map tokens to latents   
</span> <br>
tokens_to_summary_latents(tokens_TN, LOOKUP_QKD )    <br>
(First arg in (T, N) form, Second arg is datastructure mapping tokens to latents)  
<br><br>



<span style="color: blue;">
Returns latents for a single cobook level of codes 
</span> <br>
token_level_to_latents(tokens_TN: torch.Tensor, level_q: int, LOOKUP_QKD: torch.Tensor)    
<br><br>

<span style="color: blue;">
Encodes an audio signal to tokens
</span> <br>
encode_audio_to_tokens(audio, model, DEVICE, n_q_to_bandwidth(8))  
<br><br>

<span style="color: blue;">
Returns latents for an audio signal
</span> <br>
audio_to_latents(audio, model, device: str, bandwidth)  
<br><br>


<span style="color: blue;">
(T,N) tensor of latents to audio  
</span> <br>
latents128_to_audio(model, z_T128, device)  
<br><br>

<span style="color: blue;">
(T,N) tensor of tokens to audio  
</span> <br>
tokens_TN_to_audio_1T(model, tokens_TN: torch.Tensor, device, audio_scales=None, last_frame_pad_length: int = 0)  
<br><br>



In [3]:

if 0: # Load the "pool" from an.ecdc file (tokens_TN is (T, n_q)) ----
    dspath = "ecdc/DSPeepers--max_range-00.55--c-02--x-98.ecdc"
    pool_tokens_TN, scales, raw = load_ecdc(dspath)
else : # OR from audio 
    #dspath = "wav24k/amen_mono_24k.wav" # "wav24k/diverse24.wav"
    dspath = "wav24k/bells24.wav"
    pool_audio=load_wav_mono(dspath)
    pool_tokens_TN, raw = encode_audio_to_tokens(pool_audio, model, DEVICE, n_q_to_bandwidth(8))
    scales = getattr(raw, "audio_scales", None)


encode_audio_to_tokens with n_q=8


In [4]:
#show some infor about the pool
n_q= pool_tokens_TN.shape[1]
print("pool_tokens_TN:", pool_tokens_TN.shape, pool_tokens_TN.dtype)   # (T, n_q)
print("n_q:", n_q)
print("scales:", scales)
print("raw keys:", list(raw.keys()))
print("audio_length:", raw.get("audio_length"))

pool_tokens_TN: torch.Size([4241, 8]) torch.int64
n_q: 8
scales: [None]
raw keys: ['audio_codes', 'audio_scales', 'last_frame_pad_length']
audio_length: None


<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>Create the lookup table for using in function taking codes -> latents </b>

In [5]:
# Create lookup table (with same number of codebooks as poos) for fast token->latent mapping
K = int(getattr(model.config, "codebook_size", 1024))
n_q_data = int(pool_tokens_TN.shape[1])
LOOKUP_QKD = build_LOOKUP_via_layer_decode(model, n_q=n_q_data, K=K, device=DEVICE)
print("LOOKUP_QKD:", LOOKUP_QKD.shape, LOOKUP_QKD.dtype, LOOKUP_QKD.device)

LOOKUP_QKD: torch.Size([8, 1024, 128]) torch.float32 cpu


<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>POOL  latents</b>

In [6]:
# now use the lookup table to first create the "pool" of latents
pool_latents = tokens_to_summary_latents(pool_tokens_TN, LOOKUP_QKD)
print("pool_latents:", pool_latents.shape, pool_latents.dtype, pool_latents.device)


Taking tokens_to_summary_latents with n_q=8
pool_latents: torch.Size([4241, 128]) torch.float32 cpu


In [7]:
pool_latents_audio = latents128_to_audio(model, pool_latents, DEVICE).cpu()
display(Audio(pool_latents_audio, rate=24000))

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>TARGET latents</b>

In [8]:
target_audio=load_wav_mono("wav24k/amen_mono_24k.wav")
print(f'DEVICE is {DEVICE}')
target_latents, _, _ = audio_to_latents(target_audio, model, DEVICE, n_q_to_bandwidth(n_q))
print("target_latents:", target_latents.shape, target_latents.dtype, target_latents.device)

# lets just make sure when we convert the latents back to audio it sound right.
test_target_audio= latents128_to_audio(model, target_latents, DEVICE).cpu()
display(Audio(test_target_audio, rate=24000))

DEVICE is cpu
encode_audio_to_tokens with n_q=8
target_latents: torch.Size([524, 128]) torch.float32 cpu


<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>Visualization / Exploration </b>

